<a href="https://colab.research.google.com/github/marcodiaz01/Predicci-n-de-Churn-uso-de-MLflow-y-FastAPI/blob/main/ModeloChurnFastAPI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Despliegue de Modelo de Churn con FastAPI


**Instalar dependencias y preparar el entorno**

FastAPI normalmente requiere un servidor web (uvicorn). Para correrlo dentro de un entorno bloqueante como Google Colab y poder probarlo en tiempo real, usaremos las librerías nest_asyncio y requests (o curl).

In [15]:
# 1. Instalar FastAPI, Uvicorn y Pydantic
!pip install fastapi uvicorn pydantic --quiet

# 2. Configurar parches de ejecución asíncrona para Colab
import nest_asyncio
import uvicorn
from threading import Thread

nest_asyncio.apply()
print("✅ Entorno e instalaciones preparadas correctamente.")


✅ Entorno e instalaciones preparadas correctamente.


# Paso 2: Crear el Servidor, Esquemas, Monitoreo y Endpoints

En este bloque escribiremos el código de la API. Incluiremos:

• Pydantic: Para validar que los datos del cliente entren con el formato correcto.

• Logging: Registros básicos en consola para simular el monitoreo (saber cuándo entra una solicitud y si hubo errores).

• Predicción Simulada: Con la lógica de nuestro modelo ganador.

In [17]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import logging
import random

# 1. Configurar el sistema de Monitoreo Básico (Logging)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - [SISTEMA MONITOREO] - %(message)s"
)
logger = logging.getLogger("ChurnAPI")

# 2. Inicializar la aplicación FastAPI
app = FastAPI(title="Servicio de Predicción de Churn de Streaming")

# 3. Definir la estructura obligatoria de los datos de entrada
class ClienteData(BaseModel):
    horas_visualizacion: float = Field(..., description="Horas vistas al mes", ge=0)
    antiguedad_meses: int = Field(..., description="Meses con la suscripción", ge=0)
    edad: int = Field(..., description="Edad del suscriptor", ge=18)

# Endpoint de verificación de estado (Health Check)
@app.get("/")
def home():
    logger.info("Solicitud recibida en el endpoint de verificación (Health Check).")
    return {"status": "Online", "modelo_activo": "RandomForest_Variante_2"}

# Endpoint para recibir los datos y realizar la predicción de Churn
@app.post("/predict")
def predict_churn(cliente: ClienteData):
    logger.info(f"Procesando predicción para cliente. Edad: {cliente.edad}, Antigüedad: {cliente.antiguedad_meses} meses.")

    try:
        # Lógica simulada basada en el modelo ganador (Variante 2)
        if cliente.horas_visualizacion < 15 and cliente.antiguedad_meses <= 3:
            probabilidad = random.uniform(0.75, 0.95)
            prediccion = "Cancela Suscripción (Churn)"
        else:
            probabilidad = random.uniform(0.05, 0.30)
            prediccion = "Se Mantiene Activo"

        logger.info(f"Resultado enviado con éxito: {prediccion}")

        return {
            "prediccion": prediccion,
            "probabilidad_churn": round(probabilidad, 4),
            "modelo_version": "RandomForest_V2_Medio"
        }

    except Exception as e:
        logger.error(f"Se ha detectado un fallo en el servicio: {str(e)}")
        raise HTTPException(status_code=500, detail="Error interno del modelo")

# 4. Función para inicializar Uvicorn en segundo plano
def run_server():
    # Desactivamos los logs por defecto de uvicorn para ver únicamente nuestro sistema de monitoreo personalizado
    uvicorn.run(app, host="127.0.0.1", port=8000, log_config=None)

# Lanzar el servidor en un hilo independiente para que Colab no se quede congelado
server_thread = Thread(target=run_server, daemon=True)
server_thread.start()
print("🚀 Servidor FastAPI encendido exitosamente y escuchando en http://127.0.0.1:8000")



🚀 Servidor FastAPI encendido exitosamente y escuchando en http://127.0.0.1:8000


ERROR:uvicorn.error:[Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use



# Paso 3: Probar la API con Curl (Verificación de solicitudes)

Ya que el servidor está corriendo en segundo plano dentro de Colab, podemos simular que somos una aplicación externa (o Postman) enviando solicitudes HTTP usando comandos curl.

In [19]:
# Prueba 1: Verificar el estado general de la API (Debe retornar status: Online)
!curl -s http://127.0.0.1:8000/

# Imprimir un separador visual en la consola de Colab
!echo -e "\n\n--- Enviando datos de un Cliente con Riesgo de Churn ---"

# Prueba 2: Predicción enviando los parámetros en formato JSON al puerto y ruta correctos
!curl -s -X POST "http://127.0.0.1:8000/predict" \
     -H "Content-Type: application/json" \
     -d '{"horas_visualizacion": 5.2, "antiguedad_meses": 2, "edad": 24}'

{"status":"Online","modelo_activo":"RandomForest_Variante_2"}

--- Enviando datos de un Cliente con Riesgo de Churn ---
{"prediccion":"Cancela Suscripción (Churn)","probabilidad_churn":0.8629,"modelo_versión":"RandomForest_V2_Medio"}

In [22]:
# Prueba 1: Verificar el estado general de la API (Debe retornar status: Online)
!curl -s http://127.0.0.1:8000/

# Imprimir un separador visual en la consola de Colab
!echo -e "\n\n--- Enviando datos de un Cliente con Riesgo de Churn ---"

# Prueba 2: Predicción enviando los parámetros en formato JSON al puerto y ruta correctos
!curl -s -X POST "http://127.0.0.1:8000/predict" \
     -H "Content-Type: application/json" \
     -d '{"horas_visualizacion": 120.5, "antiguedad_meses": 14, "edad": 35}'

{"status":"Online","modelo_activo":"RandomForest_Variante_2"}

--- Enviando datos de un Cliente con Riesgo de Churn ---
{"prediccion":"Se Mantiene Activo","probabilidad_churn":0.2856,"modelo_versión":"RandomForest_V2_Medio"}